# 07 · Tableau Export

Tableau Public can't connect to SQL Server, so this notebook writes clean CSVs to `tableau/data/` for the three dashboard pages:

1. **Forecast accuracy:** actual vs LightGBM and naive by store-week (from Step 5)
2. **Promo impact:** matched promo lift by store and weekday (same design as Step 6)
3. **Inventory planner:** safety stock and reorder points by store across service levels (same method as Step 8)

Each file is built so ratios aggregate correctly in Tableau: WAPE = SUM(abs error) / SUM(sales), lift = SUM(promo sales) / SUM(control sales) − 1. A check after each section reproduces the headline numbers from the earlier steps.

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import pyodbc
from scipy.stats import norm

BACKTEST = "../data/processed/backtest_predictions.csv"
OUT_DIR  = "../tableau/data"
os.makedirs(OUT_DIR, exist_ok=True)

# Must match Step 8
REVIEW_WEEKS, LEAD_WEEKS = 1, 1
SERVICE_LEVELS = [0.80, 0.85, 0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995]

CLUSTER_NAMES = {1: "Weekday/promo-sensitive", 2: "Summer/tourist", 3: "Christmas-driven",
                 4: "Sunday-trading", 5: "Balanced week"}
DAY_NAMES = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

def cluster_name(s):
    num = pd.to_numeric(s, errors="coerce")
    return num.map(CLUSTER_NAMES) if num.notna().all() else s.astype(str)

## 2. Forecast accuracy

In [2]:
bt = pd.read_csv(BACKTEST, parse_dates=["WeekStart"])
bt = bt[(bt["WeeklySales"] > 0) & (bt["Pred_LGBM"] > 0) & (bt["Pred_Naive"] > 0)].copy()
bt["ClusterName"] = cluster_name(bt["Cluster"])

acc = bt[["Window", "Store", "WeekStart", "ClusterName", "WeeklySales", "Pred_LGBM", "Pred_Naive"]].copy()
for m in ["LGBM", "Naive"]:
    acc[f"AbsErr_{m}"] = (acc["WeeklySales"] - acc[f"Pred_{m}"]).abs()
acc.to_csv(f"{OUT_DIR}/forecast_accuracy.csv", index=False, float_format="%.2f")

wape = lambda m: acc[f"AbsErr_{m}"].sum() / acc["WeeklySales"].sum()
print(f"Saved {len(acc):,} store-weeks | WAPE LightGBM {wape('LGBM'):.1%}, naive {wape('Naive'):.1%}  (Step 5: about 6.3% vs 11.2%, small differences expected)")

Saved 49,819 store-weeks | WAPE LightGBM 6.3%, naive 11.2%  (Step 5: about 6.3% vs 11.2%, small differences expected)


## 3. Promo impact

In [3]:
conn = pyodbc.connect("Driver={ODBC Driver 18 for SQL Server};Server=localhost;"
                      "Database=RetailForecast;Trusted_Connection=yes;TrustServerCertificate=yes;")
df = pd.read_sql("""
    SELECT d.Store, d.SalesDate AS [Date], d.DayOfWeek, d.Sales, d.Customers, d.Promo,
           d.StateHoliday, c.Cluster
    FROM daily_sales d JOIN store_clusters c ON c.Store = d.Store
    WHERE d.IsOpen = 1 AND d.Sales > 0""", conn, parse_dates=["Date"])
conn.close()
df["Promo"] = df["Promo"].astype(int)
df["Hol"] = (df["StateHoliday"].astype(str).str.strip() != "0").astype(int)
print(df.shape)

C:\Users\lkaus\AppData\Local\Temp\ipykernel_2388\2845449334.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


(844338, 9)


In [4]:
# Same matched design as Step 6: promo day vs same store and weekday at t-7 and t+7 (non-promo),
# excluding state holidays and December
cols = ["Sales", "Customers", "Promo", "Hol"]
def shifted(days, sfx):
    x = df[["Store", "Date"] + cols].copy()
    x["Date"] += pd.Timedelta(days=days)
    return x.rename(columns={c: c + sfx for c in cols})

m = (df.merge(shifted(7, "_prev"), on=["Store", "Date"])
       .merge(shifted(-7, "_next"), on=["Store", "Date"]))
m = m[(m["Promo"] == 1) & (m["Promo_prev"] == 0) & (m["Promo_next"] == 0)
      & (m[["Hol", "Hol_prev", "Hol_next"]].sum(axis=1) == 0) & (m["Date"].dt.month != 12)].copy()
for v in ["Sales", "Customers"]:
    m[f"{v}_ctrl"] = (m[f"{v}_prev"] + m[f"{v}_next"]) / 2

lift = (m.groupby(["Store", "Cluster", "DayOfWeek"])
          .agg(Pairs=("Sales", "size"), PromoSales=("Sales", "sum"), CtrlSales=("Sales_ctrl", "sum"),
               PromoCustomers=("Customers", "sum"), CtrlCustomers=("Customers_ctrl", "sum"))
          .reset_index())
lift["ClusterName"] = cluster_name(lift["Cluster"])
lift["DayName"] = lift["DayOfWeek"].map(DAY_NAMES)
lift = lift.drop(columns="Cluster")
lift.to_csv(f"{OUT_DIR}/promo_lift.csv", index=False, float_format="%.2f")

print(f"Saved {len(lift):,} store-weekday rows from {len(m):,} matched pairs (Step 6: 223,945)")
print(f"Overall lift {lift['PromoSales'].sum() / lift['CtrlSales'].sum() - 1:+.1%}  (Step 6: +37.1%)")

Saved 5,575 store-weekday rows from 223,945 matched pairs (Step 6: 223,945)
Overall lift +37.1%  (Step 6: +37.1%)


## 4. Inventory planner

In [5]:
# Same method as Step 8 (normal), repeated for each service level so Tableau can use a slider
for m_ in ["LGBM", "Naive"]:
    bt[f"Err_{m_}"] = (bt["WeeklySales"] - bt[f"Pred_{m_}"]) / bt[f"Pred_{m_}"]
latest = bt[bt["Window"] == sorted(bt["Window"].unique())[-1]]
H = REVIEW_WEEKS + LEAD_WEEKS

frames = []
for sl in SERVICE_LEVELS:
    out = None
    for m_ in ["LGBM", "Naive"]:
        g = bt.groupby("ClusterName")[f"Err_{m_}"]
        bias, sd = g.mean(), g.std()
        p = latest.groupby(["Store", "ClusterName"])[f"Pred_{m_}"].mean().rename("WeeklyForecast").reset_index()
        bc, per_week = p["ClusterName"].map(bias), norm.ppf(sl) * p["ClusterName"].map(sd) * p["WeeklyForecast"]
        p = p.assign(**{
            f"WeeklyForecast_{m_}": p["WeeklyForecast"],
            f"SafetyStock_{m_}":    per_week * np.sqrt(H),
            f"ReorderPoint_{m_}":   p["WeeklyForecast"] * (1 + bc) * LEAD_WEEKS + per_week * np.sqrt(LEAD_WEEKS),
            f"OrderUpTo_{m_}":      p["WeeklyForecast"] * (1 + bc) * H + per_week * np.sqrt(H),
        }).drop(columns="WeeklyForecast")
        out = p if out is None else out.merge(p, on=["Store", "ClusterName"])
    out["ServiceLevel"] = sl
    frames.append(out)
inv = pd.concat(frames, ignore_index=True)
inv.to_csv(f"{OUT_DIR}/inventory_service_levels.csv", index=False, float_format="%.3f")

at95 = inv[inv["ServiceLevel"] == 0.95]
print(f"Saved {len(inv):,} rows ({inv['Store'].nunique():,} stores x {len(SERVICE_LEVELS)} service levels)")
print(f"At 95%: LightGBM €{at95['SafetyStock_LGBM'].sum()/1e6:.1f}M, naive €{at95['SafetyStock_Naive'].sum()/1e6:.1f}M  (Step 8: €9.0M vs €19.7M)")

Saved 12,265 rows (1,115 stores x 11 service levels)
At 95%: LightGBM €9.0M, naive €19.7M  (Step 8: €9.0M vs €19.7M)


## 5. Files written

In [6]:
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(f"{OUT_DIR}/{f}") / 1e6
    print(f"{f:32s} {size:5.1f} MB")

forecast_accuracy.csv              4.4 MB
inventory_service_levels.csv       1.3 MB
promo_lift.csv                     0.4 MB
